In [1]:
!pip install -q -U torchao pylcs faiss-cpu rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.7 MB/s eta 0:00:00


In [2]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

Mounted at /content/drive
STATE RESTORED: 36501 corpus | 2618 LLM answers | all caches loaded


In [4]:
# =============================================================================
# CELL 1 — ENG_UGA HYBRID SELECTOR (free test on existing e3 probe generations)
# Mechanic: add the generated answer to the candidate pool as one more candidate,
# let consensus utility decide per query. Tune gen's pseudo-sim on 150, confirm on 150.
# =============================================================================
import json
import numpy as np

pa = json.load(open(OUTPUT_DIR / 'ftqwen_e3_Eng_Uga_probe.json'))
probe = [int(k) for k in pa.keys()]
tune, conf = probe[:150], probe[150:]
tag, a, m = choice['Eng_Uga']

def hybrid_score(idx_list, gen_sim_mode):
    """gen_sim_mode: 'max' (gen as confident as top-1) or 'mean3' (average of top-3 sims)"""
    tot = []
    for i in idx_list:
        cands = list(val_cands_all[i])
        sims = [c['sim'] for c in cands]
        gsim = max(sims) if gen_sim_mode == 'max' else float(np.mean(sims[:3]))
        ext = cands + [{'answer': pa[str(i)], 'sim': gsim, 'idx': -1}]
        dd, ddw, u1, uL = uni_prep(ext)
        ans = dd[mbr_idx(0.5*u1 + 0.5*uL, ddw, a, m)]
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        at = uni_toks(ans)[:CAP]
        tot.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
    return np.mean(tot)

def base_score(idx_list):
    tot = []
    for i in idx_list:
        dd, ddw, u1, uL = uni_prep(val_cands_all[i])
        ans = dd[mbr_idx(0.5*u1 + 0.5*uL, ddw, a, m)]
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        at = uni_toks(ans)[:CAP]
        tot.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
    return np.mean(tot)

best_mode = max(['max', 'mean3'], key=lambda gm: hybrid_score(tune, gm))
h_conf = hybrid_score(conf, best_mode)
b_conf = base_score(conf)
print(f"Eng_Uga hybrid [{best_mode}]: confirm-half hybrid={h_conf:.4f}  comb_mbr={b_conf:.4f}  "
      f"Δ={h_conf-b_conf:+.4f}")
print("GATE: Δ > +0.003 -> scale up (generate full Eng_Uga val+test). Δ <= 0 -> dead, zero cost.")

Eng_Uga hybrid [max]: confirm-half hybrid=0.6365  comb_mbr=0.6365  Δ=+0.0000
GATE: Δ > +0.003 -> scale up (generate full Eng_Uga val+test). Δ <= 0 -> dead, zero cost.


In [5]:
# =============================================================================
# PER-LANGUAGE RETRIEVER FT: Eng_Uga + Lug_Uga (answer-similarity supervision)
# One model per language. Eval: that language only, via interpolation grid.
# GPU: T4/L4 fine (8-bit optim). ~45 min train per language + eval.
# =============================================================================
import torch, gc, random, json
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
random.seed(42); np.random.seed(42)

AFRIE5_DIR = OUTPUT_DIR / 'afrie5-final-model'
PREFIX = "query: "
TARGETS = ['Eng_Uga', 'Lug_Uga']
train_q_set = set(q.strip() for q in train_df['input'].dropna().astype(str))
is_train_row = np.array([q in train_q_set for q in corpus_q_stripped])

def mine_lang(sub, cap=6000):
    """Triplets from answer-similarity: pos = neighbor w/ answer R1>=0.5;
       neg = closer-ranked neighbor w/ answer R1<=0.2. Train rows only."""
    idx_t = [i for i in range(len(combined)) if subsets_raw[i]==sub and is_train_row[i]]
    random.shuffle(idx_t); idx_t = idx_t[:cap]
    ix, mask = lang_indices[sub]; mask_arr = np.array(mask)
    tmask = np.array([is_train_row[ci] for ci in mask])
    trips = []
    for qi in tqdm(idx_t, desc=f"mine {sub}", leave=False):
        gold = uni_toks(answers_raw[qi])[:CAP]
        if len(gold) < 3: continue
        D, I = ix.search(corpus_emb[qi].reshape(1,-1), 25)
        cands = []
        for d, li in zip(D[0], I[0]):
            if li < 0 or not tmask[int(li)]: continue
            ci = int(mask_arr[int(li)])
            if ci == qi or corpus_q_stripped[ci] == corpus_q_stripped[qi]: continue
            cands.append((ci, float(d), uni_r1(gold, uni_toks(answers_raw[ci])[:CAP])))
        pos = [c for c in cands if c[2] >= 0.5]
        if not pos: continue
        best = max(pos, key=lambda c: c[2])
        negs = [c for c in cands if c[1] >= best[1] - 0.02 and c[2] <= 0.2][:2]
        for ng in negs:
            trips.append((f"{PREFIX}{questions_raw[qi]}",
                          f"{PREFIX}{questions_raw[best[0]]}",
                          f"{PREFIX}{questions_raw[ng[0]]}"))
    return trips

results = {}
for sub in TARGETS:
    trips = mine_lang(sub)
    print(f"{sub}: {len(trips)} triplets")
    if len(trips) < 500:
        print(f"  -> supply too thin, skipping {sub}"); continue
    # ---- train (one language only) ----
    gc.collect(); torch.cuda.empty_cache()
    model = SentenceTransformer(str(AFRIE5_DIR) if AFRIE5_DIR.exists()
            else 'McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
    model.max_seq_length = 96
    model[0].auto_model.gradient_checkpointing_enable()
    import bitsandbytes as bnb
    loader = DataLoader([InputExample(texts=list(t)) for t in trips],
                        shuffle=True, batch_size=8, drop_last=True)
    model.fit(train_objectives=[(loader, losses.MultipleNegativesRankingLoss(model))],
              epochs=1, warmup_steps=int(0.1*len(loader)),
              optimizer_class=bnb.optim.AdamW8bit, optimizer_params={'lr': 8e-6},
              use_amp=True, show_progress_bar=True)
    out_dir = OUTPUT_DIR / f'afrie5-{sub.lower()}'
    model.save(str(out_dir))
    # ---- encode this language's corpus slice + its val queries ----
    _, mask = lang_indices[sub]
    enc = lambda T: model.encode([f"{PREFIX}{t}" for t in T], batch_size=64,
            normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    sub_corpus = enc([questions_raw[ci] for ci in mask])
    np.save(CACHE/f'pl_{sub}_corpus.npy', sub_corpus)
    v_idx = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub]
    sub_val = enc([val_qs[i] for i in v_idx])
    np.save(CACHE/f'pl_{sub}_val.npy', sub_val)
    vmap = {i: j for j, i in enumerate(v_idx)}
    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

    # ---- eval: interpolation grid, split-half, this language only ----
    mask_arr = np.array(mask)
    tag, a, m = choice[sub]
    idxs = [i for i in v_idx if val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    tune, hold = idxs[0::2][:300], idxs[1::2][:300]
    def pool_at(i, beta, k=15):
        s = beta*(corpus_emb[mask_arr] @ val_emb[i]) + (1-beta)*(sub_corpus @ sub_val[vmap[i]])
        order = np.argsort(-s); out = []
        q = val_qs[i].strip()
        for j in order[:k+5]:
            ci = int(mask_arr[j])
            if corpus_q_stripped[ci] == q: continue
            out.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
            if len(out) >= k: break
        return out
    def wsc(ix, beta):
        tot = []
        for i in ix:
            pool = pool_at(i, beta)
            if not pool: continue
            dd, ddw, u1, uL = uni_prep(pool)
            ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]; at = uni_toks(ans)[:CAP]
            tot.append(0.37*uni_r1(rt, at)+0.37*uni_rl(rt, at))
        return np.mean(tot)
    cur = wsc(hold, 1.0)
    bb = max([0.8,0.6,0.4,0.2,0.0], key=lambda b: wsc(tune, b))
    new = wsc(hold, bb)
    results[sub] = (cur, new, bb)
    print(f"{sub}: cur={cur:.4f}  per-lang@β={bb}: {new:.4f}  Δ={new-cur:+.4f}  "
          f"{'ADOPT' if new > cur + 0.002 else 'reject'}")
print("\nFinal:", results)

Eng_Uga: 942 triplets


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Lug_Uga: cur=0.6111  per-lang@β=0.8: 0.6172  Δ=+0.0061  ADOPT

Final: {'Eng_Uga': (np.float64(0.6395287930947541), np.float64(0.6391371453438739), 0.0), 'Lug_Uga': (np.float64(0.611131957872719), np.float64(0.6172112593668362), 0.8)}


In [4]:
!pip install -q -U torchao pylcs bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00


In [6]:
# ==== V5 PREP: encode Lug_Uga test queries with the per-language model ====
from sentence_transformers import SentenceTransformer
import numpy as np, gc, torch
m = SentenceTransformer(str(OUTPUT_DIR/'afrie5-lug_uga'), device='cuda:0')
m.max_seq_length = 96
t_idx = [i for i in range(len(test_df)) if test_subs[i]=='Lug_Uga']
emb = m.encode([f"query: {test_qs[i]}" for i in t_idx], batch_size=64,
               normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
np.save(CACHE/'pl_Lug_Uga_test.npy', emb)
json.dump(t_idx, open(CACHE/'pl_Lug_Uga_test_idx.json','w'))
m.cpu(); del m; gc.collect(); torch.cuda.empty_cache()
print(f"Encoded {len(t_idx)} Lug_Uga test queries")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Encoded 374 Lug_Uga test queries


In [7]:
# =============================================================================
# DIAGNOSTIC PROBES: per-language LB decomposition (3 submissions)
# Probe A: Eng_Uga + Lug_Uga real | rest "-"
# Probe B: Aka_Gha + Eng_Gha real | rest "-"
# Probe C: Swa_Ken + Eng_Ken real | rest "-"
# =============================================================================
import pandas as pd
v4 = pd.read_csv(OUTPUT_DIR / 'submission_v4_final.csv')
id2sub = dict(zip(test_df['ID'].astype(str), test_subs))

PROBES = {'probeA_uga': {'Eng_Uga','Lug_Uga'},
          'probeB_gha': {'Aka_Gha','Eng_Gha'},
          'probeC_ken': {'Swa_Ken','Eng_Ken'}}
for name, langs in PROBES.items():
    p = v4.copy()
    mask = p['ID'].astype(str).map(id2sub).isin(langs)
    for col in ['TargetR1F1','TargetRLF1','TargetLLM']:
        p.loc[~mask, col] = "-"
    p.to_csv(OUTPUT_DIR / f'submission_{name}.csv', index=False)
    print(f"{name}: {mask.sum()} real answers, {(~mask).sum()} blanked")

# decoder: run after all three scores are in
def decode(probe_scores):
    """probe_scores: {'probeA_uga': (r1, rl, llm), ...} from the LB breakdowns.
       Recovers per-language-pair public contribution and implied per-language score."""
    n_pub = {s: None for s in PROBES}  # public counts unknown; use test shares as estimate
    test_share = test_df['subset'].value_counts(normalize=True)
    for name, langs in PROBES.items():
        share = sum(test_share.get(l, 0) for l in langs)
        r1, rl, llm = probe_scores[name]
        print(f"{name} ({'+'.join(langs)}, ~{100*share:.1f}% of test):")
        print(f"  implied within-slice scores: R1≈{r1/share:.4f} RL≈{rl/share:.4f} LLM≈{llm/share:.4f}")

probeA_uga: 1118 real answers, 1500 blanked
probeB_gha: 983 real answers, 1635 blanked
probeC_ken: 396 real answers, 2222 blanked


In [8]:
# sim predictions per language (for the truth-vs-sim table)
for sub in ['Eng_Uga','Lug_Uga','Aka_Gha','Eng_Gha','Swa_Ken','Eng_Ken']:
    print(sub, "sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers")

Eng_Uga sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers
Lug_Uga sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers
Aka_Gha sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers
Eng_Gha sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers
Swa_Ken sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers
Eng_Ken sim wR (per V4 strategy): see RESULTS tables — paste alongside probe numbers


In [9]:
# ==== PROBE B': Gha slice, Aka_Gha -> comb_mbr (stitch removed), Eng_Gha unchanged ====
import pandas as pd, json
from tqdm import tqdm
v4 = pd.read_csv(OUTPUT_DIR / 'submission_v4_final.csv')
id2sub = dict(zip(test_df['ID'].astype(str), test_subs))
p = v4.copy()
mask_gha = p['ID'].astype(str).map(id2sub).isin({'Aka_Gha','Eng_Gha'})
for col in ['TargetR1F1','TargetRLF1','TargetLLM']:
    p.loc[~mask_gha, col] = "-"
# replace Aka_Gha rows with comb_mbr answers
aka_rows = [i for i in range(len(test_df)) if test_subs[i]=='Aka_Gha']
tag, a, m = choice['Aka_Gha']
new_ans = {}
for i in tqdm(aka_rows, desc="Aka comb_mbr"):
    pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], 'Aka_Gha',
                                    k=K_CANDIDATES, exclude_exact=False)
    if not pool: continue
    dd, ddw, u1, uL = uni_prep(pool)
    new_ans[str(test_df.iloc[i]['ID'])] = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
sel = p['ID'].astype(str).isin(new_ans)
for col in ['TargetR1F1','TargetRLF1','TargetLLM']:
    p.loc[sel, col] = p.loc[sel, 'ID'].astype(str).map(new_ans)
p.to_csv(OUTPUT_DIR / 'submission_probeB2_gha.csv', index=False)
print(f"Probe B': {mask_gha.sum()} real ({len(new_ans)} Aka swapped to comb_mbr)")

Aka comb_mbr: 100%|██████████| 492/492 [00:05<00:00, 84.64it/s]


Probe B': 983 real (492 Aka swapped to comb_mbr)


In [10]:
# ==== HyDE PROBE: answer-to-answer retrieval using cached FT generations ====
import json, numpy as np, torch, gc
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

GENS = {'Eng_Uga': 'ftqwen_e3_Eng_Uga_probe.json', 'Lug_Uga': 'ftqwen_luguga_val.json',
        'Eng_Gha': 'ftqwen_enggha_val.json', 'Aka_Gha': 'ftqwen_akagha_val.json'}
bienc = SentenceTransformer(str(OUTPUT_DIR/'afrie5-final-model'), device='cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"{'Sub':<10} {'n':>5} {'cur top1 wR':>11} {'HyDE top1 wR':>12} {'blend best':>10} {'delta':>8}")
for sub, fname in GENS.items():
    gens = json.load(open(OUTPUT_DIR/fname))
    idxs = [int(k) for k in gens if val_cands_all[int(k)]
            and str(val_df.iloc[int(k)]['output']).strip()][:300]
    hyde_emb = bienc.encode([f"passage: {gens[str(i)]}" for i in idxs], batch_size=64,
                            normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    _, mask = lang_indices[sub]; mask = np.array(mask)
    sub_ans = ans_emb[mask]                      # cached answer embeddings, this language
    sub_q   = corpus_emb[mask]
    tag, a, m = choice[sub]
    rows = {'cur': [], 'hyde': [], 'blend': []}
    for j, i in enumerate(idxs):
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        q = val_qs[i].strip()
        def top1(scores):
            for k in np.argsort(-scores)[:6]:
                ci = int(mask[k])
                if corpus_q_stripped[ci] == q: continue
                at = uni_toks(answers_raw[ci])[:CAP]
                return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
            return 0.0
        s_q = sub_q @ val_emb[i]
        s_h = sub_ans @ hyde_emb[j]
        rows['cur'].append(top1(s_q))
        rows['hyde'].append(top1(s_h))
        rows['blend'].append(max(top1(0.5*s_q + 0.5*s_h), top1(0.7*s_q + 0.3*s_h)))  # quick blend scan
    cur, hy, bl = np.mean(rows['cur']), np.mean(rows['hyde']), np.mean(rows['blend'])
    best = max(hy, bl)
    print(f"{sub:<10} {len(idxs):>5} {cur:>11.4f} {hy:>12.4f} {bl:>10.4f} {best-cur:>+8.4f}")
print("\nNOTE: 'blend' takes max over two betas on the same data — optimistic; "
      "any language showing +0.005 here gets a proper split-half before adoption.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Sub            n cur top1 wR HyDE top1 wR blend best    delta
Eng_Uga      300      0.6395       0.6345     0.6374  -0.0022
Lug_Uga      300      0.6139       0.5400     0.5696  -0.0443
Eng_Gha      300      0.2043       0.2062     0.2137  +0.0094
Aka_Gha      300      0.1977       0.2022     0.2130  +0.0153

NOTE: 'blend' takes max over two betas on the same data — optimistic; any language showing +0.005 here gets a proper split-half before adoption.
